# `generate_scm_gamma_26()`

This scenario produces synthetic panel data with one treated unit and multiple donors. It uses a low-level **Gamma DGP** to simulate realistic outcomes with time-varying exposure and latent rates.

### 1. Panel Geometry
- **Donors:** $J = 8$
- **Time:** $T = 48$ periods ($T_{pre} = 36$, $T_{post} = 12$)
- **Intervention:** Starts at $T_0 = 37$

### 2. Generating Base Components
For each unit $j$ at time $t$, we generate two main components:

**1. Log-Exposure ($E_{jt}$):**
$$ \log E_{jt} = a_j + b_j\,\tilde t + c_t + \xi_{jt} $$
- $a_j, b_j$: Unit-specific level and trend.
- $c_t, \xi_{jt}$: Common time shocks and unit noise.

**2. Latent Log-Rate ($\eta_{jt}$):**
$$ \eta_{jt} = \alpha_j + g_j\,\tilde t + s_j S_t + M_t + \lambda_j^\top F_t + u_{jt} $$
- $S_t, M_t$: Seasonality and Macro AR(1) effects.
- $F_t, u_{jt}$: Latent factors and unit noise.

The baseline mean is then: $\mu_{jt} = E_{jt}\,\exp(\eta_{jt})$.

### 3. Treated Counterfactual ($\mu_t^{(0)}$)
The treated unit's counterfactual is built as a noisy mixture of donors:
$$
E_t^{(0)} = \left(\sum w_j E_{jt}\right) \exp(\delta_t^E), \qquad
\eta_t^{(0)} = \sum w_j \eta_{jt} + \delta_t^{\eta}
$$
$$ \mu_t^{(0)} = E_t^{(0)} \exp(\eta_t^{(0)}) $$
where $w$ are fixed mixture weights and $\delta$ represents "pre-fit mismatch" noise.

### 4. Treatment Effect with Ramp-In
After $T_0$, we apply a relative lift $\tau_k^{rate}$ that ramps up over time:
$$ r_k = 1 - e^{-(k+1)/2.5}, \quad k = 0 \dots T_{post}-1 $$
$$ \mu_t^{(1)} = \mu_t^{(0)} \left(1 + (\beta_0 + \beta_1 k) r_k \right) $$

The first post-period effect is attenuated by $r_0 \approx 0.33$.

### 5. Gamma Outcome Layer
Final outcomes $Y$ are sampled to keep $\mathbb{E}[Y] = \mu$:
$$ Y \sim \text{Gamma}(\text{shape}=k, \text{scale}=\mu/k) $$
This ensures variance is proportional to $\mu^2$ (constant coefficient of variation).

### 6. Oracle Treatment Effects (ATT)
The Average Treatment Effect on the Treated (ATT) is the average impact of the intervention across all post-treatment periods. In this synthetic scenario, we can calculate it in two ways:

1. **Realized ATT**: Based on observed vs. counterfactual outcomes.
$$ \text{ATT}_{realized} = \frac{1}{T_{post}} \sum_{t \in \text{Post}} (Y_t - Y_t^{(0)}) $$
In the data, this is the mean of `tau_realized_true` for the treated unit in post-periods.

2. **Mean ATT**: Based on the underlying population means (the "signal").
$$ \text{ATT}_{mean} = \frac{1}{T_{post}} \sum_{t \in \text{Post}} (\mu_t^{(1)} - \mu_t^{(0)}) $$
In the data, this is the mean of `tau_mean_true` for the treated unit in post-periods.


In [1]:
import numpy as np
import pandas as pd

from causalis.scenarios.synthetic_control import generate_scm_gamma_26, AugmentedSyntheticControl
from causalis.dgp.panel_data_scm import generate_scm_gamma_data


In [2]:
panel = generate_scm_gamma_26(
    include_oracles=True,
    return_panel_data=True,
    seed=42,
)

panel


PanelDataSCM(df=(432, 4), unit_id='unit_id', time_id='time_id', y='y', treated_unit='treated', intervention_time=37, donor_units=['donor_1', 'donor_2', 'donor_3', 'donor_4', 'donor_5', 'donor_6', 'donor_7', 'donor_8'], observed_col='observed')

In [3]:
df = panel.df.copy()

print('shape:', df.shape)
print('units:', df['unit_id'].nunique())
print('pre periods:', len(panel.pre_times()))
print('post periods:', len(panel.post_times()))
print('columns:', list(df.columns))

df.head()


shape: (432, 4)
units: 9
pre periods: 36
post periods: 12
columns: ['unit_id', 'time_id', 'observed', 'y']


,unit_id,time_id,observed,y
0,donor_1,1,1,6.546174
1,donor_1,2,1,2.855563
2,donor_1,3,1,12.436816
3,donor_1,4,1,10.939923
4,donor_1,5,1,4.564390


In [4]:
# Oracle identity check 1
np.allclose(df['tau_realized_true'].to_numpy(), (df['y'] - df['y_cf']).to_numpy())


KeyError: 'tau_realized_true'

In [5]:
# Oracle identity check 2
np.allclose(df['tau_mean_true'].to_numpy(), (df['mu_treated'] - df['mu_cf']).to_numpy())


KeyError: 'tau_mean_true'

In [6]:
# First-post attenuation check (default treatment_effect_rate=0.12, slope=0.01)
treated = df[df['unit_id'] == panel.treated_unit].sort_values('time_id')
first_post_time = panel.post_times()[0]
first_post = treated[treated['time_id'] == first_post_time].iloc[0]

observed_rate = float(first_post['tau_mean_true'] / first_post['mu_cf'])
expected_rate = (0.12 + 0.01 * 0.0) * (1.0 - np.exp(-1.0 / 2.5))

print('first post time:', first_post_time)
print('observed relative lift:', observed_rate)
print('expected relative lift:', expected_rate)


KeyError: 'tau_mean_true'

### 7. Inspecting Structural Exposure
The wrapper `generate_scm_gamma_26` hides internal covariates. To inspect the time-varying `exposure` directly, use the low-level generator:


In [ ]:
df_low = generate_scm_gamma_data(
    n_pre_periods=24,
    n_post_periods=8,
    n_donors=8,
    return_panel_data=False,
    seed=123,
)

donor1 = df_low[df_low['unit_id'] == 'donor_1'].sort_values('time_id')
treated_low = df_low[df_low['unit_id'] == 'treated'].sort_values('time_id')

print('donor_1 exposure unique values:', donor1['exposure'].nunique())
print('treated exposure unique values:', treated_low['exposure'].nunique())

df_low[['unit_id', 'time_id', 'exposure', 'mu_cf', 'mu_treated']].head()


### 8. SCM Fit Example
Estimate the Average Treatment Effect (ATT) using ASCM and compare it against the oracle truths.


In [7]:
est = AugmentedSyntheticControl(lambda_aug=0.5).fit(panel).estimate()

post_treated = df[(df['unit_id'] == panel.treated_unit) & (df['time_id'].isin(panel.post_times()))]
oracle_att_realized = post_treated['tau_realized_true'].mean()
oracle_att_mean = post_treated['tau_mean_true'].mean()

print('Estimated ATT:', est.att)
print('Oracle ATT (realized):', oracle_att_realized)
print('Oracle ATT (mean):', oracle_att_mean)


KeyError: 'tau_realized_true'